In [1]:
import pymupdf
import pandas as pd

In [2]:
sf = "jeevan_utsav_cis.pdf"

In [3]:
dfs = []
pages = pymupdf.open(sf)

for page in pages:
    tables = page.find_tables()
    for table in tables:
        df = table.to_pandas()
        dfs.append(df)

In [4]:
def get_policy_clause(raw_clause):
    clauses = []
    for i, clause in enumerate(raw_clause):
        if raw_clause[i-1] == "Condition":
            clause = raw_clause[i-1] + " "+ clause
            clauses.remove(raw_clause[i-1])
        if len(clause) <= 3:
            clauses[-1] += " "+clause
            continue
        clauses.append(clause)
    return clauses

In [5]:
def get_section_details(clause_list, desc):
    details = ""
    txt = ""
    # print(f"\n\nInside section clause: desc: {desc}; desc length: {len(desc)}  clause length: {len(clause_list)}")
    # print(f"\n\nInside section clause: {len(clause_list)} desc: {len(desc)}")
    # print(desc)
    for i, clause in enumerate(clauses):
        if clause is None or clause == "Schedule" or clause == "":
            continue
        if isinstance(desc, list):
            txt = desc[i]
        else:
            txt = desc
            # print(desc)
        details += f"{txt}\n\nPolicy Clause Number: {clause}\n\n"
    return details

In [6]:
def get_standardized_section(raw_clause, desc):
    clauses = []
    if desc.find("\uf0b7") != -1:
        desc = desc.split("\uf0b7")[1:]
    
        for i, clause in enumerate(raw_clause):
            if raw_clause[i-1] == "Condition" or (raw_clause[i-1].split(" ")[0] == "Condition" and len(raw_clause[i-1].split(" ")[1]) <= 3) or clause.strip() == "Definitions":
                clause = raw_clause[i-1] + " "+ clause
                clauses.remove(raw_clause[i-1])
            if len(clause) <= 3:
                clauses[-1] += " "+clause
                continue
            clauses.append(clause)
    
    else:
        if len(raw_clause) == 1:
            clauses = raw_clause
        else:
            clause = ""
            # sep = ""
            for i, r in enumerate(raw_clause):
                sep = " " if i < len(raw_clause) else ""
                clause += r + sep
            
            clauses.append(clause)
    
    # print(clauses)
    return desc, clauses

In [7]:
## First page and second page incorporated

info = ""

for i, df in enumerate(dfs):
    flag = 0
    if i > 1:
        break
    if i == 1:
        df = df.T.reset_index().T.reset_index(drop=True)
        if 'Col' in df.iloc[:,0].values[0]:
            desc = dfs[i-1].iloc[-1,:].values[2]
            desc += df.iloc[:,2].values[0]
            
            clauses = dfs[i-1].iloc[-1,:].values[3].split('\n')
            clauses += df.iloc[:,3].values[0].split('\n')

            # print(f"Desc: {desc}\n\n")
            
            desc, clauses = get_standardized_section(clauses, desc)
            details = get_section_details(clauses, desc)
            
            info += f"## {dfs[i-1].iloc[-1,:].values[1]}\n\n{details}"
            flag = 1

    for idx, row in df.iterrows():
        if flag == 1:
            flag = 0
            continue
        
        details = ""
        if idx == df.index[-1]:
            continue
        
        desc = row.values[2]
        
        clauses = row.values[3].split('\n')
        # clauses = row.values[clause_col_id].split('\n')

        desc, clauses = get_standardized_section(clauses, desc)

        # if desc.find("\uf0b7") != -1:
        #     desc = desc.split("\uf0b7")[1:]
        
        #     clauses = get_policy_clause(clauses)
        details = get_section_details(clauses, desc)

        info += f"## {row.values[1]}\n\n{details}"

print(info)

## Name of the Insurance
Product
and Unique
Identification Number
(UIN)

LIC’s Jeevan Utsav
(UIN: 512N363V02)

Policy Clause Number: Part A

## Policy Number

_____________________

Policy Clause Number: Part A

## Type of Insurance
Policy

Non-Par, Non-Linked, Individual, Savings, Whole
Life Insurance Plan

Policy Clause Number: Part B - Definitions 

## Basic Policy details

 Sum Assured on Death: Sum Assured on
Death is defined as Higher of Basic Sum
Assured or 7 times of Annualized Premium

Policy Clause Number: Condition 1.A of Part C

## Riders opted, if any

## Policy Coverage /
benefits payable

 Benefit payable on Death:
On death of the Life Assured after the date of
commencement of risk, Death Benefit equal to
“Sum Assured on Death” along with accrued
Guaranteed Additions shall be payable,provided the policy is in-force.
This Death Benefit shall not be less than 105%
of total premiums paid upto the date of death.


Policy Clause Number: Condition 1.A of Part C

 Survival Bene